# 추가 봇 데이터 결합과 재학습 입력

원본 최종 수정 기록: 2025-06-06T00:35:25+09:00.

파일명의 번호는 파이프라인 순서이며 버전 번호가 아닙니다. 데이터·모델 경로와 실행 환경을 설정한 뒤 필요한 단계를 실행하세요.


In [ ]:
import pandas as pd
from sklearn.preprocessing import StandardScaler

In [ ]:
df_1 = pd.read_csv('x_user_data.csv')

In [ ]:
df_2 = pd.read_csv('Bot_user_data2.csv')

In [ ]:
retrain_com_df = pd.concat([df_1, df_2], ignore_index=True)

In [ ]:
label_col = 'fake'
features = retrain_com_df.drop(columns=[label_col])
label = retrain_com_df[label_col]

scaler = StandardScaler()
scaler.fit(features)
normalized_features = scaler.transform(features)
df_normalized = pd.DataFrame(normalized_features, columns=features.columns)

df_normalized[label_col] = label
df_normalized.to_csv('retrain_z_score.csv', index=False)

print("완료")

In [ ]:
df_3 = pd.read_csv('train.csv')

In [ ]:
df_4 = pd.read_csv("test.csv")

In [ ]:
df_com = pd.concat([df_3,df_4], ignore_index = True)

In [ ]:
label_col = 'fake'
test_features = df_com.drop(columns=[label_col])
test_label = df_com[label_col]

test_normalized_features = scaler.transform(test_features)
test_df_normal = pd.DataFrame(test_normalized_features, columns = test_features.columns)


In [ ]:
test_df_normal[label_col] = test_label

In [ ]:
test_df_normal.to_csv('model_b_prime_test.csv')

In [ ]:
df_x =  df_normalized

In [ ]:
from scipy.stats import ttest_ind

# 봇과 휴먼 계정으로 나누기
bot_df = df_x[df_x['fake'] == 1]
human_df = df_x[df_x['fake'] == 0]

# t-test 수행
t_test_results = []
for col in df_x.columns[:-1]:  # 'fake' 제외
    t_stat, p_value = ttest_ind(bot_df[col], human_df[col], equal_var=False)
    t_test_results.append({
        "feature": col,
        "t_statistic": t_stat,
        "p_value": p_value,
        "reject_null": p_value < 0.05
    })

# 결과를 DataFrame으로 변환하고 정렬
t_test_df = pd.DataFrame(t_test_results).sort_values(by="p_value")

t_test_df


In [ ]:

t_test_df.to_csv("model_b_prime_data_vali.csv", index=False)
